# EDA: Online Retail II

Quick exploratory analysis of the cleaned & aggregated daily e-commerce metrics.

**Sections:**
1. Dataset overview
2. Daily metrics time series
3. Seasonal decomposition (revenue)
4. Distributions & boxplots
5. Correlation matrix
6. Conclusions

In [ ]:
import matplotlib.pyplot as plt
import pandas as pd
import seaborn as sns
from statsmodels.tsa.seasonal import seasonal_decompose

sns.set_theme(style="whitegrid", font_scale=1.1)
plt.rcParams["figure.figsize"] = (14, 5)
plt.rcParams["figure.dpi"] = 100

In [ ]:
# Load aggregated daily metrics (run `python -m scripts.preprocess` first)
daily = pd.read_csv("../data/processed/daily_metrics.csv", index_col="date", parse_dates=True)

# Exclude missing days (store closed) from analysis where needed
active = daily[~daily["missing_day"].astype(bool)].copy()
print(f"Total days: {len(daily)}, active trading days: {len(active)}")

## 1. Dataset overview

In [ ]:
print(f"Period: {daily.index.min().date()} to {daily.index.max().date()}")
print(f"Days total: {len(daily)}")
print(f"Active trading days: {len(active)}")
print(f"Missing (closed) days: {daily['missing_day'].astype(bool).sum()}")
print()
active[["revenue", "orders", "avg_check", "unique_customers", "items_sold"]].describe().round(1)

In [ ]:
# Country breakdown from raw data
raw = pd.read_csv("../data/raw/online_retail_II.csv", usecols=["Country"])
country_counts = raw["Country"].value_counts()
print(f"Unique countries: {len(country_counts)}")
print()
print("Top 10 countries by transaction count:")
print(country_counts.head(10).to_string())

## 2. Daily metrics time series

In [ ]:
def plot_metric(metric: str, title: str):
    """Plot daily metric with weekend shading."""
    fig, ax = plt.subplots(figsize=(14, 4))

    # Shade weekends
    for d in daily[daily["is_weekend"].astype(bool)].index:
        ax.axvspan(d, d + pd.Timedelta(days=1), color="lightgray", alpha=0.3)

    ax.plot(active.index, active[metric], linewidth=0.8, color="steelblue")
    ax.set_title(title)
    ax.set_xlabel("Date")
    ax.set_ylabel(metric)
    plt.tight_layout()
    plt.show()

plot_metric("revenue", "Daily Revenue")
plot_metric("orders", "Daily Orders")

In [ ]:
# Revenue by day of week
fig, axes = plt.subplots(1, 2, figsize=(14, 4))

dow_labels = ["Mon", "Tue", "Wed", "Thu", "Fri", "Sat", "Sun"]

active.groupby("day_of_week")["revenue"].mean().plot.bar(
    ax=axes[0], color="steelblue"
)
axes[0].set_xticklabels(dow_labels, rotation=0)
axes[0].set_title("Mean Revenue by Day of Week")
axes[0].set_ylabel("Revenue")

active.groupby("month")["revenue"].mean().plot.bar(
    ax=axes[1], color="steelblue"
)
axes[1].set_title("Mean Revenue by Month")
axes[1].set_ylabel("Revenue")

plt.tight_layout()
plt.show()

## 3. Seasonal decomposition (revenue)

In [ ]:
# Use full daily index (with 0-filled missing days) for decomposition
result = seasonal_decompose(daily["revenue"], model="additive", period=7)

fig, axes = plt.subplots(4, 1, figsize=(14, 10), sharex=True)
result.observed.plot(ax=axes[0], title="Observed", linewidth=0.7)
result.trend.plot(ax=axes[1], title="Trend", linewidth=0.7, color="orange")
result.seasonal.plot(ax=axes[2], title="Seasonal (period=7)", linewidth=0.7, color="green")
result.resid.plot(ax=axes[3], title="Residual", linewidth=0.7, color="red")
plt.tight_layout()
plt.show()

In [ ]:
# Large residuals are candidate anomalies
resid = result.resid.dropna()
threshold = resid.std() * 3
outlier_dates = resid[resid.abs() > threshold]
print(f"Residuals beyond 3-sigma ({threshold:,.0f}):")
for d, v in outlier_dates.items():
    print(f"  {d.date()}  residual={v:>+12,.0f}  revenue={daily.loc[d, 'revenue']:>10,.0f}")

## 4. Distributions & boxplots

In [ ]:
metrics = ["revenue", "orders", "avg_check", "items_sold"]

fig, axes = plt.subplots(2, 4, figsize=(16, 7))

for i, m in enumerate(metrics):
    # Histogram
    axes[0, i].hist(active[m], bins=40, color="steelblue", edgecolor="white")
    axes[0, i].set_title(f"{m} distribution")

    # Boxplot
    axes[1, i].boxplot(active[m], vert=True)
    axes[1, i].set_title(f"{m} boxplot")

plt.tight_layout()
plt.show()

In [ ]:
# IQR-based outlier counts per metric
print("IQR outlier counts (active days only):")
for m in metrics:
    q1, q3 = active[m].quantile(0.25), active[m].quantile(0.75)
    iqr = q3 - q1
    lo, hi = q1 - 1.5 * iqr, q3 + 1.5 * iqr
    n_out = ((active[m] < lo) | (active[m] > hi)).sum()
    print(f"  {m:>20s}: {n_out} outliers  (range {lo:>10,.0f} .. {hi:>10,.0f})")

## 5. Correlation matrix

In [ ]:
corr_cols = ["revenue", "orders", "avg_check", "unique_customers",
             "items_sold", "avg_items_per_order", "unique_products"]

corr = active[corr_cols].corr()

fig, ax = plt.subplots(figsize=(8, 7))
sns.heatmap(corr, annot=True, fmt=".2f", cmap="coolwarm", center=0,
            square=True, linewidths=0.5, ax=ax)
ax.set_title("Metric Correlation Matrix")
plt.tight_layout()
plt.show()

## 6. Conclusions

**Seasonality:**
- Clear **weekly** cycle: weekends (especially Saturday) show significantly lower revenue and orders.
- **Monthly** pattern: revenue peaks in Oct-Nov (pre-holiday season), drops sharply in late December-January.
- Seasonal decomposition confirms a strong 7-day period in the revenue series.

**Visible anomalies:**
- Several extreme revenue spikes are visible in the time series (the decomposition residuals flag them beyond 3-sigma).
- `avg_check` has the most right-skewed distribution with heavy outliers — a few days had very large single orders.
- `items_sold` also shows occasional massive spikes (bulk orders).

**Correlations:**
- `revenue`, `orders`, `unique_customers`, and `unique_products` are strongly correlated (they all measure "how busy the day was").
- `avg_check` and `avg_items_per_order` are less correlated with volume — they capture order-level behavior, not day-level volume.

**Implications for anomaly detection:**
- Detectors must account for weekly seasonality (Saturday dips are normal, not anomalies).
- Revenue and orders are the best primary metrics for detection; avg_check is useful as a secondary signal.
- The 135 missing days (store closed) should be excluded from detection, not treated as zero-anomalies.